# MiniMind `selfrewrite` — A100 Reproduction & Distillation Data Harvesting

Run top-to-bottom on a **Google Colab A100 (40 GB / 80 GB)** runtime.
Reproduces gradient-free Self-Rewriting (`arXiv:2611.21885v2`) and harvests verified traces into `distill_bundle/v1/` on Google Drive for model downsizing (`27B -> 9B -> 4B -> 2B -> MiniMind`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi

In [ ]:
%cd /content
!if [ -d minimind/.git ]; then cd minimind && git fetch origin mychanges && git checkout mychanges && git pull --ff-only origin mychanges; else git clone --depth 1 -b mychanges https://github.com/ed2k/minimind.git; fi
%cd /content/minimind
!pip -q install "transformers>=5.0.0" "peft>=0.14.0" "datasets>=3.6.0" "huggingface_hub" "safetensors" "accelerate" "bitsandbytes" "sympy" "torchvision" "pillow"
!pip -q install flash-linear-attention causal-conv1d || echo "Using reference GatedDeltaNet fallback"
!python3 -m unittest discover -s selfrewrite/tests -v

In [ ]:
# Batch 1: Reproduce Qwen3.5-27B Self-Improvement (Protocol 2) & Self-Repair on A100
!python3 -m selfrewrite.colab.colab_launcher --workspace-dir selfrewrite/workspace --mode reproduce --backend hf --model-id Qwen/Qwen3.5-27B --profile auto --drive-dir /content/drive/MyDrive/minimind_selfrewrite --num-reproduce-improve 8

In [ ]:
# Batch 2: Harvest Disjoint Distillation Traces & Build Leak-Audited distill_bundle/v1/
!python3 -m selfrewrite.colab.colab_launcher --workspace-dir selfrewrite/workspace --mode harvest --backend hf --model-id Qwen/Qwen3.5-27B --profile auto --fast-harvest-profile --drive-dir /content/drive/MyDrive/minimind_selfrewrite --num-harvest-improve 8 --num-harvest-repair 12
# For long A100 runs via colab-mcp/browser without cell timeout, run detached & poll HEARTBEAT.json:
# !nohup python3 -u -m selfrewrite.colab.colab_launcher --workspace-dir selfrewrite/workspace --mode harvest --backend hf --model-id Qwen/Qwen3.5-27B --profile auto --fast-harvest-profile --drive-dir /content/drive/MyDrive/minimind_selfrewrite --num-harvest-improve 8 --num-harvest-repair 12 > /content/drive/MyDrive/minimind_selfrewrite/live.log 2>&1 &
# !cat selfrewrite/workspace/HEARTBEAT.json && tail -n 20 /content/drive/MyDrive/minimind_selfrewrite/live.log
!cat selfrewrite/workspace/distill_bundle/v1/quality_audit.json
!cat selfrewrite/workspace/report.md

In [ ]:
# Batch 3: Downsizing Student Bracket (4B -> 2B -> 9B -> 0.6B) & Frozen Zero-Gradient Eval
!python3 -m selfrewrite.colab.colab_launcher --workspace-dir selfrewrite/workspace --mode bracket --backend hf --rungs R2,R3,R1,R4 --profile auto --drive-dir /content/drive/MyDrive/minimind_selfrewrite --bootstrap-next-rung
!cat selfrewrite/workspace/downsize_report.md